# Etapa 3 — Modelagem

Entrada: `../data/processed/dataset.csv`, da Etapa 2.
Saída: um classificador treinado, avaliado e interpretado, e um ranking dos
melhores horários da janela.

Um detalhe que define esta etapa: **não existe label pronta.** O site
classifica as condições para pesca, não para surf — e pesca gosta de mar
calmo, quase o oposto. Definir o que é uma condição boa para surfar é a
primeira tarefa aqui, e é uma decisão sua.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

DIR_PROC = Path("..") / "data" / "processed"

In [3]:
SEED = 42 
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Carregar o dataset processado

Confira que veio o que você espera: número de linhas, tipos, nulos. Se a
Etapa 2 mudou, isso aparece aqui.

In [4]:
df = pd.read_csv(DIR_PROC / "dataset.csv", parse_dates=["datahora"])

print("Formato:", df.shape)
print("Período:", df["datahora"].min(), "→", df["datahora"].max())
print("Nulos:", df.isna().sum().sum())
display(df.dtypes)
df.head()

Formato: (168, 14)
Período: 2026-10-08 00:00:00 → 2026-10-14 23:00:00
Nulos: 0


datahora         datetime64[us]
data                        str
altura_onda_m           float64
direcao_onda                str
vento_kmh               float64
direcao_vento               str
altura_mare_m           float64
mare_subindo               bool
coeficiente               int64
idade_lua                 int64
hora_dia                  int64
dia_semana                  str
periodo                     str
luz_do_dia                 bool
dtype: object

,datahora,data,altura_onda_m,direcao_onda,vento_kmh,direcao_vento,altura_mare_m,mare_subindo,coeficiente,idade_lua,hora_dia,dia_semana,periodo,luz_do_dia
0,2026-10-08 00:00:00,2026-10-08,1.1,E,21.0,SE,1.85,True,91,27,0,qui,madrugada,False
1,2026-10-08 01:00:00,2026-10-08,1.1,E,22.0,SE,2.23,True,91,27,1,qui,madrugada,False
2,2026-10-08 02:00:00,2026-10-08,1.1,E,20.0,SE,2.40,True,91,27,2,qui,madrugada,False
3,2026-10-08 03:00:00,2026-10-08,1.1,E,18.0,SSE,2.31,False,91,27,3,qui,madrugada,False
4,2026-10-08 04:00:00,2026-10-08,1.1,E,18.0,SSE,1.99,False,91,27,4,qui,madrugada,False


## 2. Definir a label

Escreva, em código, a regra que separa condições ruins, boas e ótimas (ou
outro conjunto de classes, se preferir).

- Quais variáveis entram na regra: altura da onda, vento, maré, direção?
- Quais limiares? De onde vem o número — leitura, conversa com surfista,
  distribuição dos próprios dados? Registre a origem, mesmo que seja um chute
  informado.
- Como as classes ficaram distribuídas? Se 95% das horas caírem numa classe,
  a métrica do modelo vai mentir para você.

Documente a regra em markdown antes de seguir. Tudo depois disso depende dela.

## Regra de rótulo

Só considero horas com luz do dia, porque ninguém surfa à noite. Restam 91 horas.

Cada hora ganha pontos:

| Condição | Pontos |
|---|---|
| Onda >= 1,2 m | +2 |
| Onda >= 1,1 m | +1 |
| Vento <= 20 km/h | +2 |
| Vento <= 23 km/h | +1 |
| Maré entre 0,8 e 2,0 m | +1 |

- **ótima:** 4 ou 5 pontos
- **boa:** 2 ou 3 pontos
- **ruim:** 0 ou 1 ponto

**De onde vêm os números:** onda maior é melhor; vento mais fraco é melhor, porque ele sopra do mar e bagunça a onda; meia-maré evita recife exposto e onda quebrando na areia. Os limiares são chutes informados, ajustados à distribuição desta semana. Por isso a regra compara as horas desta janela entre si, mas não diz se o mar está bom em termos absolutos.

In [ ]:
def pontuar(linha):
    pontos = 0
    if linha["altura_onda_m"] >= 1.2:
        pontos += 2
    elif linha["altura_onda_m"] >= 1.1:
        pontos += 1
    if linha["vento_kmh"] <= 20:
        pontos += 2
    elif linha["vento_kmh"] <= 23:
        pontos += 1
    if 0.8 <= linha["altura_mare_m"] <= 2.0:
        pontos += 1
    return pontos

def rotular(linha):
    if linha["pontos"] >= 4:
        return "ótima"
    if linha["pontos"] >= 2:
        return "boa"
    return "ruim"

ORDEM = ["ruim", "boa", "ótima"]

df = df[df["luz_do_dia"]].copy()  # só horas com luz: ninguém surfa à noite
df["pontos"] = df.apply(pontuar, axis=1)
df["condicao"] = pd.Categorical(df.apply(rotular, axis=1), categories=ORDEM, ordered=True)

print(df["condicao"].value_counts())
print(df["condicao"].value_counts(normalize=True).round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="condicao", order=ORDEM, ax=ax[0])
ax[0].set_title("Distribuição das classes")
pd.crosstab(df["data"], df["condicao"]).plot(kind="bar", stacked=True, ax=ax[1])
ax[1].set_title("Classes por dia")
plt.tight_layout()
plt.show()

## 3. Features e alvo

Separe `X` e `y`.

- Quais colunas entram como feature? Cuidado: usar diretamente as variáveis da
  sua regra faz o modelo apenas redescobrir o `if` que você escreveu. Isso é
  esperado aqui — mas você precisa saber que é isso que está acontecendo.
- Direção do vento e da onda são texto. Como transformar em número sem inventar
  uma ordem que não existe?

### Escolhas

**Entram:** as três variáveis da regra (`altura_onda_m`, `vento_kmh`, `altura_mare_m`) e mais `mare_subindo`, `coeficiente`, `hora_dia` e as direções de onda e vento. O modelo vai redescobrir o `if` da regra, e isso é esperado. As variáveis extras servem para ver se ele passa a usar algo que a regra não usa.

**Ficam de fora:**

- `pontos` e `condicao`: são a própria resposta. Usá-las seria vazamento.
- `datahora`, `data`, `dia_semana`: identificam o dia, e o modelo poderia decorar "dia 14 é bom" em vez de aprender sobre o mar.
- `periodo`: repete a informação de `hora_dia`.
- `luz_do_dia`: depois do filtro, é sempre `True`.
- `idade_lua`: repete o `coeficiente` (os dois têm um valor só por dia).

**Direções:** viram one-hot, ou seja, uma coluna 0/1 para cada direção. Trocar `E`, `ESE`, `SE` por 1, 2, 3 inventaria uma ordem e uma distância entre elas que não existem.

**Escala:** `StandardScaler` nas numéricas, porque o KNN mede distância e, sem escala, o vento (13–28) pesaria mais que a onda (0,9–1,4) só por ter números maiores. A árvore não precisa disso, mas também não atrapalha.

In [ ]:
FEATURES_NUM = ["altura_onda_m", "vento_kmh", "altura_mare_m", "mare_subindo", "coeficiente", "hora_dia"]
FEATURES_CAT = ["direcao_onda", "direcao_vento"]

X = df[FEATURES_NUM + FEATURES_CAT].copy()
X["mare_subindo"] = X["mare_subindo"].astype(int)
y = df["condicao"]

preprocessador = ColumnTransformer([
    ("num", StandardScaler(), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

print("X:", X.shape, "| y:", y.shape)
X.head()

## 4. Treino e teste

Divida os dados. Vale estratificar pelas classes? A ordem temporal importa —
sortear linhas de um mesmo dia entre treino e teste vaza informação?

### Escolhas

**Separo por dia, não por hora.** Horas vizinhas são quase idênticas (a onda muda 0,1 m em várias horas). Se eu sorteasse horas, cada hora do teste teria uma "gêmea" no treino e a nota sairia inflada.

**Não dá para fazer um split puramente temporal** (treinar em 08–12 e testar em 13–14). 17 das 23 horas `ótima` estão em 13 e 14/out, e esses dias não têm nenhuma hora `ruim`: o treino quase não veria `ótima` e o teste não teria `ruim`.

**Escolha:** teste com os dias **11 e 14/out**, que juntos têm as três classes; treino com os outros 5 dias. Escolhi os dias olhando os rótulos, e declaro isso. A escolha faz o papel da estratificação: estratificar por classe direto, linha a linha, quebraria os dias.

São só 26 horas de teste, então uma nota nesse split sozinha não é confiável. Na seção 6, complemento com validação cruzada por dia (`GroupKFold`).

In [7]:
DIAS_TESTE = ["2026-10-11", "2026-10-14"]
eh_teste = df["data"].isin(DIAS_TESTE)

X_treino, X_teste = X[~eh_teste], X[eh_teste]
y_treino, y_teste = y[~eh_teste], y[eh_teste]

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)
pd.DataFrame({"treino": y_treino.value_counts(), "teste": y_teste.value_counts()})

NameError: name 'X' is not defined

## 5. Treinar

Treine pelo menos dois modelos de famílias diferentes (por exemplo uma árvore
e um modelo baseado em distância) para ter com o que comparar.

## 6. Avaliar

- Acurácia é suficiente aqui, com as classes desbalanceadas como estão?
- Olhe a matriz de confusão: que erro o modelo comete?
- Um único split é confiável com poucas linhas, ou vale validação cruzada?

## 7. Interpretar

Um modelo que acerta e você não sabe por quê não serve para responder nada.

- Plote a árvore. Os cortes que ela encontrou parecem com os seus limiares?
- Quais features o modelo considera mais importantes? Bate com sua intuição?
- Se a acurácia veio altíssima, explique o motivo — não celebre.

## 8. Responder a pergunta

Aplique o modelo na janela inteira e produza a resposta para Felipe e Nicholas:

- Quais os melhores dias da janela de previsão?
- Em média, qual o melhor horário do dia?
- Em que grau essa resposta é uma previsão do mar, e em que grau é apenas a
  sua regra reescrita?

## 9. Conclusão e limites

Em texto:

1. A recomendação, em uma frase.
2. O teto deste modelo: ele reproduz a sua heurística, então é tão bom quanto
   ela. O que faria dela uma regra melhor?
3. O que a janela de ~7 dias e o volume de linhas impedem de afirmar.
4. Como você responderia "qual a melhor época do ano para surfar em João
   Pessoa" — que dado faltaria, e como coletá-lo.